# Create analysis samples from the full dataset

The full file `02_merged_data_full.parquet` is too large to load into memory with `pd.read_parquet` (it raises `ArrowMemoryError`). This notebook builds smaller **random samples** for analysis without ever holding the whole dataset in memory.

**How it stays within memory:** instead of reading the file at once, we **stream it in batches** with PyArrow. For each batch we draw one random number per row, and a row is kept for a given sample if that number falls below the sample's fraction. The kept rows are written straight to the output parquet as we go. At any moment only one batch (`BATCH_SIZE` rows) plus its sampled subsets is in RAM — so peak memory is roughly constant regardless of how large the source file is.

We build **two samples in a single streaming pass**:

| fraction | output file |
|---|---|
| 50% | `02_merged_data_sample.parquet` |
| 10% | `02_merged_data_sample_10pct.parquet` |

Because both samples share the *same* per-row random draw, the 10% sample is a strict subset of the 50% sample. Everything is reproducible (fixed `SEED`); the analysis notebooks read these samples instead of the full file.

In [1]:
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

SOURCE = "../data/02_merged_data_full.parquet"

# Each output is built in the same streaming pass. Map fraction -> output path.
SAMPLES = {
    0.5: "../data/02_merged_data_sample.parquet",        # keep ~50% of the rows
    0.1: "../data/02_merged_data_sample_10pct.parquet",  # keep ~10% of the rows
}

SEED = 42              # makes the samples reproducible
BATCH_SIZE = 200_000   # rows read into memory at a time (controls peak memory)

In [2]:
# Inspect the file via its metadata only — this does NOT read the data into memory.
pf = pq.ParquetFile(SOURCE)
meta = pf.metadata

print(f"{meta.num_rows:,} rows")
print(f"{meta.num_columns} columns")
print(f"{meta.num_row_groups} row groups")
for fraction, output in SAMPLES.items():
    print(f"~{round(meta.num_rows * fraction):,} rows expected in {output} (fraction={fraction})")

13,505,728 rows
92 columns
110 row groups
~6,752,864 rows expected in ../data/02_merged_data_sample.parquet (fraction=0.5)
~1,350,573 rows expected in ../data/02_merged_data_sample_10pct.parquet (fraction=0.1)


## Stream, sample, and write

We iterate the file batch by batch. For each batch we draw **one** random number per row, then keep that row for every sample whose fraction it falls below, appending the kept rows to that sample's output parquet through its own `ParquetWriter`. Sharing the single draw across fractions makes the smaller sample a strict subset of the larger one. The full dataset (and each full sample) is never materialised in memory at once.

In [3]:
rng = np.random.default_rng(SEED)

writers = {fraction: None for fraction in SAMPLES}
kept_rows = {fraction: 0 for fraction in SAMPLES}
total_rows = 0

try:
    for batch in pf.iter_batches(batch_size=BATCH_SIZE):
        n = batch.num_rows
        total_rows += n

        # One uniform draw per row, shared across all fractions so smaller
        # samples are strict subsets of larger ones.
        draw = rng.random(n)

        for fraction, output in SAMPLES.items():
            sampled = batch.filter(pa.array(draw < fraction))
            kept_rows[fraction] += sampled.num_rows

            table = pa.Table.from_batches([sampled], schema=pf.schema_arrow)
            if writers[fraction] is None:
                writers[fraction] = pq.ParquetWriter(output, table.schema)
            writers[fraction].write_table(table)
finally:
    for writer in writers.values():
        if writer is not None:
            writer.close()

print(f"read   {total_rows:,} rows")
for fraction, output in SAMPLES.items():
    kept = kept_rows[fraction]
    print(f"kept   {kept:,} rows ({kept / total_rows:.1%}) -> {output}")

read   13,505,728 rows
kept   6,755,150 rows (50.0%) -> ../data/02_merged_data_sample.parquet
kept   1,349,173 rows (10.0%) -> ../data/02_merged_data_sample_10pct.parquet


In [4]:
# Verify: each sample is small enough to load normally for analysis.
import pandas as pd

for fraction, output in SAMPLES.items():
    sample = pd.read_parquet(output)
    print(f"{output}: {sample.shape[0]:,} rows x {sample.shape[1]} columns")

sample.head()

../data/02_merged_data_sample.parquet: 6,755,150 rows x 92 columns
../data/02_merged_data_sample_10pct.parquet: 1,349,173 rows x 92 columns


,trip_id,taxi_id,trip_start,trip_end,trip_seconds,trip_miles,pickup_census_tract,dropoff_census_tract,pickup_community_area,dropoff_community_area,...,dropoff_poi_tourism_tract,dropoff_poi_nature_tract,dropoff_poi_healthcare_community,dropoff_poi_education_community,dropoff_poi_amenity_community,dropoff_poi_leisure_community,dropoff_poi_shops_community,dropoff_poi_public_services_community,dropoff_poi_tourism_community,dropoff_poi_nature_community
0,6ee8b984bf5c6ea4b8b6de7779f080a882214726,24471dbdad65718d90f3f55715fe1c45b75169615d6403...,2024-01-01 00:45:00-06:00,2024-01-01 00:57:34-06:00,0 days 00:12:34,2.57,NaN,NaN,8,7.0,...,NaN,NaN,62.0,32.0,312.0,253.0,246.0,12.0,147.0,424.0
1,3a91da5cee395df5df3afab71304fd7a5902e420,04b96cbbdcfe5b7cbb6884bc1b922819466f652662ead8...,2024-01-01 00:45:00-06:00,2024-01-01 01:07:12-06:00,0 days 00:22:12,2.67,1.703132e+10,1.703133e+10,32,33.0,...,41.0,378.0,23.0,10.0,72.0,101.0,34.0,7.0,51.0,601.0
2,461ae4214962e58a165e3b4e48bd1a54f98f7a83,643bb93d2492e6eda6e928f41ec8e1cb906c3d8bf0dbd9...,2024-01-01 00:15:00-06:00,2024-01-01 00:25:55-06:00,0 days 00:10:55,3.20,NaN,NaN,8,7.0,...,NaN,NaN,62.0,32.0,312.0,253.0,246.0,12.0,147.0,424.0
3,aa65afad59435bc4e81358e393295cb0dc487600,fb0ce19e30e712c77c57cfdb6ef729c2d2ad73225d9ec3...,2024-01-01 00:00:00-06:00,2024-01-01 00:15:29-06:00,0 days 00:15:29,7.03,NaN,NaN,8,77.0,...,NaN,NaN,31.0,13.0,139.0,90.0,133.0,6.0,24.0,258.0
4,f98233a9aa7955e778c5784f0262974ac3faf559,9054e449dca78c2363d025823dfceb7e33bebc5ca24c92...,2024-01-01 00:45:00-06:00,2024-01-01 01:07:24-06:00,0 days 00:22:24,1.36,NaN,NaN,8,28.0,...,NaN,NaN,104.0,60.0,498.0,390.0,294.0,48.0,85.0,1342.0
